#**Cliente de LLM asincrono**

| Característica | OpenAI | Anthropic | Gemini |
|----------------|---------|-----------|---------|
| **SDK async** | `AsyncOpenAI` | `AsyncAnthropic` | `genai.Client(...).aio` |
| **Método de chat** | `client.chat.completions.create(...)` | `client.messages.create(...)` | `client.aio.models.generate_content(...)` |
| **Dónde está el texto** | `response.choices[0].message.content` | `response.content[0].text` | `response.text` |
| **Streaming** | `stream=True` + `async for` | `messages.stream(...)` *(context manager)* | `generate_content_stream(...)` + `async for` |
| **Rol "system"** | Mensaje con `role="system"` | Parámetro `system` separado | `system_instruction` en la configuración, separado de `contents` |
| **Rol del asistente** | `"assistant"` | `"assistant"` | `"model"` *(nombre distinto)* |
| **¿Requiere tarjeta?** | Sí | Sí | No *(Free Tier disponible)* |

## **Instalación de dependencias**

In [ ]:
!pip install -q openai anthropic google-genai pydantic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.4 MB/s eta 0:00:00


## **Configuración de las API keys**

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("🔑 OPENAI_API_KEY: ").strip()

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("🔑 ANTHROPIC_API_KEY: ").strip()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("🔑 Ingresá tu GOOGLE_API_KEY (gratis en aistudio.google.com/apikey): ").strip()

🔑 OPENAI_API_KEY: ··········
🔑 ANTHROPIC_API_KEY: ··········
🔑 Ingresá tu GOOGLE_API_KEY (gratis en aistudio.google.com/apikey): ··········


## **schemas.py: validación con Pydantic**

In [ ]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field, SecretStr, field_validator


class Provider(str, Enum):
    OPENAI = "openai"
    ANTHROPIC = "anthropic"
    GEMINI = "gemini"


class ChatMessage(BaseModel):
    role: str = Field(description="'user', 'assistant' o 'system'")
    content: str

    @field_validator("role")
    @classmethod
    def rol_valido(cls, v: str) -> str:
        roles_permitidos = {"user", "assistant", "system"}
        if v not in roles_permitidos:
            raise ValueError(f"role debe ser uno de {roles_permitidos}, recibido: '{v}'")
        return v


class LLMConfig(BaseModel):
    provider: Provider
    model: str
    openai_api_key: Optional[SecretStr] = None
    anthropic_api_key: Optional[SecretStr] = None
    google_api_key: Optional[SecretStr] = None
    temperature: float = Field(default=0.7, ge=0, le=2)
    max_tokens: int = Field(default=1024, gt=0)


class ModelResponse(BaseModel):
    provider: Provider
    model: str
    content: str
    error: Optional[str] = None

In [ ]:
from pydantic import ValidationError

try:
    LLMConfig(provider=Provider.OPENAI, model="gpt-4o-mini", temperature=5)
except ValidationError as e:
    print("❌ Se detectó ANTES de llamar a la API:\n", e)

❌ Se detectó ANTES de llamar a la API:
 1 validation error for LLMConfig
temperature
  Input should be less than or equal to 2 [type=less_than_equal, input_value=5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## **BaseLLMClient: la clase base abstracta**

Qué error evita: el acoplamiento. Si mañana el negocio pide "agregá Gemini", solo se crea un GeminiClient nuevo que respete este contrato — nada del resto del código cambia.

In [ ]:
from abc import ABC, abstractmethod
from typing import AsyncGenerator, List


class BaseLLMClient(ABC):
    """Contrato que todo cliente de LLM debe cumplir, sin importar el proveedor real detrás."""

    @abstractmethod
    async def generate(self, messages: List[ChatMessage]) -> ModelResponse:
        """Genera una respuesta completa (modo normal, no streaming)."""
        raise NotImplementedError

    @abstractmethod
    async def generate_stream(self, messages: List[ChatMessage]) -> AsyncGenerator[str, None]:
        """Genera la respuesta token a token (modo streaming)."""
        raise NotImplementedError
        yield  # nunca se ejecuta; solo le indica a Python que este método es un generador

## **OpenAIClient**

Qué error evita: el "bloqueo del Event Loop" — usar await en cada llamada de red asegura que, mientras OpenAI "piensa", el programa puede seguir atendiendo otras tareas.

In [ ]:
from openai import AsyncOpenAI, APIError, RateLimitError, APIConnectionError


class OpenAIClient(BaseLLMClient):
    def __init__(self, api_key: str, model: str, temperature: float, max_tokens: int):
        self._client = AsyncOpenAI(api_key=api_key)
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens

    async def generate(self, messages: List[ChatMessage]) -> ModelResponse:
        try:
            response = await self._client.chat.completions.create(
                model=self.model,
                messages=[m.model_dump() for m in messages],
                temperature=self.temperature,
                max_tokens=self.max_tokens,
            )
            return ModelResponse(
                provider=Provider.OPENAI,
                model=self.model,
                content=response.choices[0].message.content,
            )
        except RateLimitError as e:
            return ModelResponse(provider=Provider.OPENAI, model=self.model, content="",
                                  error=f"Límite de cuota excedido: {e}")
        except APIConnectionError as e:
            return ModelResponse(provider=Provider.OPENAI, model=self.model, content="",
                                  error=f"Error de conexión: {e}")
        except APIError as e:
            return ModelResponse(provider=Provider.OPENAI, model=self.model, content="",
                                  error=f"Error de la API de OpenAI: {e}")

    async def generate_stream(self, messages: List[ChatMessage]) -> AsyncGenerator[str, None]:
        try:
            stream = await self._client.chat.completions.create(
                model=self.model,
                messages=[m.model_dump() for m in messages],
                temperature=self.temperature,
                max_tokens=self.max_tokens,
                stream=True,
            )
            async for chunk in stream:
                delta = chunk.choices[0].delta.content
                if delta:
                    yield delta
        except (RateLimitError, APIConnectionError, APIError) as e:
            yield f"\n[⚠️ Error durante el streaming: {e}]"

> Punto de discusión en clase: notá que ninguna excepción de OpenAI se deja "escapar" — la fuga de excepciones (mencionada como error común en la consigna) rompería el loop principal de cualquier app que use este cliente. Acá siempre devolvemos un ModelResponse (o un chunk de texto) con el error adentro, nunca un crash.

## **AnthropicClient**

In [ ]:
from anthropic import (
    AsyncAnthropic,
    APIError as AnthropicAPIError,
    RateLimitError as AnthropicRateLimitError,
    APIConnectionError as AnthropicConnectionError,
)


class AnthropicClient(BaseLLMClient):
    def __init__(self, api_key: str, model: str, temperature: float, max_tokens: int):
        self._client = AsyncAnthropic(api_key=api_key)
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens

    async def generate(self, messages: List[ChatMessage]) -> ModelResponse:
        try:
            response = await self._client.messages.create(
                model=self.model,
                max_tokens=self.max_tokens,  # obligatorio en Anthropic, a diferencia de OpenAI
                temperature=self.temperature,
                messages=[m.model_dump() for m in messages],
            )
            return ModelResponse(
                provider=Provider.ANTHROPIC,
                model=self.model,
                content=response.content[0].text,
            )
        except AnthropicRateLimitError as e:
            return ModelResponse(provider=Provider.ANTHROPIC, model=self.model, content="",
                                  error=f"Límite de cuota excedido: {e}")
        except AnthropicConnectionError as e:
            return ModelResponse(provider=Provider.ANTHROPIC, model=self.model, content="",
                                  error=f"Error de conexión: {e}")
        except AnthropicAPIError as e:
            return ModelResponse(provider=Provider.ANTHROPIC, model=self.model, content="",
                                  error=f"Error de la API de Anthropic: {e}")

    async def generate_stream(self, messages: List[ChatMessage]) -> AsyncGenerator[str, None]:
        try:
            async with self._client.messages.stream(
                model=self.model,
                max_tokens=self.max_tokens,
                temperature=self.temperature,
                messages=[m.model_dump() for m in messages],
            ) as stream:
                async for texto in stream.text_stream:
                    yield texto
        except (AnthropicRateLimitError, AnthropicConnectionError, AnthropicAPIError) as e:
            yield f"\n[⚠️ Error durante el streaming: {e}]"

## **GeminiClient**

In [ ]:
from google import genai
from google.genai import types


class GeminiClient(BaseLLMClient):
    def __init__(self, api_key: str, model: str, temperature: float, max_tokens: int):
        self._client = genai.Client(api_key=api_key)
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens

    def _convertir_mensajes(self, messages: List[ChatMessage]):
        """Gemini separa el system prompt del resto, y llama 'model' al rol del asistente."""
        contents = []
        system_instruction = None
        for m in messages:
            if m.role == "system":
                system_instruction = m.content
            else:
                rol_gemini = "model" if m.role == "assistant" else "user"
                contents.append(types.Content(role=rol_gemini, parts=[types.Part(text=m.content)]))
        return contents, system_instruction

    async def generate(self, messages: List[ChatMessage]) -> ModelResponse:
        try:
            contents, system_instruction = self._convertir_mensajes(messages)
            response = await self._client.aio.models.generate_content(
                model=self.model,
                contents=contents,
                config=types.GenerateContentConfig(
                    temperature=self.temperature,
                    max_output_tokens=self.max_tokens,
                    system_instruction=system_instruction,
                ),
            )
            return ModelResponse(provider=Provider.GEMINI, model=self.model, content=response.text)
        except Exception as e:
            return ModelResponse(provider=Provider.GEMINI, model=self.model, content="",
                                  error=f"Error de la API de Gemini: {e}")

    async def generate_stream(self, messages: List[ChatMessage]) -> AsyncGenerator[str, None]:
        try:
            contents, system_instruction = self._convertir_mensajes(messages)
            stream = await self._client.aio.models.generate_content_stream(
                model=self.model,
                contents=contents,
                config=types.GenerateContentConfig(
                    temperature=self.temperature,
                    max_output_tokens=self.max_tokens,
                    system_instruction=system_instruction,
                ),
            )
            async for chunk in stream:
                if chunk.text:
                    yield chunk.text
        except Exception as e:
            yield f"\n[⚠️ Error durante el streaming: {e}]"

## **AsyncLLMManager (Factory Pattern)**

Qué error evita: instanciar OpenAIClient() o AnthropicClient() directamente en la lógica de negocio. Con este Factory, cambiar de proveedor es cambiar un solo campo de configuración (provider=Provider.ANTHROPIC), no reescribir código.

In [ ]:
class AsyncLLMManager:
    def __init__(self, config: LLMConfig):
        self.config = config
        self._client: BaseLLMClient = self._crear_cliente()

    def _crear_cliente(self) -> BaseLLMClient:
        if self.config.provider == Provider.OPENAI:
            if not self.config.openai_api_key:
                raise ValueError("Falta openai_api_key en la configuración")
            return OpenAIClient(
                api_key=self.config.openai_api_key.get_secret_value(),
                model=self.config.model,
                temperature=self.config.temperature,
                max_tokens=self.config.max_tokens,
            )

        if self.config.provider == Provider.ANTHROPIC:
            if not self.config.anthropic_api_key:
                raise ValueError("Falta anthropic_api_key en la configuración")
            return AnthropicClient(
                api_key=self.config.anthropic_api_key.get_secret_value(),
                model=self.config.model,
                temperature=self.config.temperature,
                max_tokens=self.config.max_tokens,
            )

        if self.config.provider == Provider.GEMINI:
            if not self.config.google_api_key:
                raise ValueError("Falta google_api_key en la configuración")
            return GeminiClient(
                api_key=self.config.google_api_key.get_secret_value(),
                model=self.config.model,
                temperature=self.config.temperature,
                max_tokens=self.config.max_tokens,
            )

        raise ValueError(f"Proveedor no soportado: {self.config.provider}")

    async def generate(self, messages: List[ChatMessage]) -> ModelResponse:
        return await self._client.generate(messages)

    async def generate_stream(self, messages: List[ChatMessage]) -> AsyncGenerator[str, None]:
        async for chunk in self._client.generate_stream(messages):
            yield chunk

## **Prueba en modo normal (OpenAI)**

In [ ]:
from pydantic import SecretStr

config_openai = LLMConfig(
    provider=Provider.OPENAI,
    model="gpt-4o-mini",
    openai_api_key=SecretStr(os.environ["OPENAI_API_KEY"]),
    temperature=0.7,
    max_tokens=200,
)

manager_openai = AsyncLLMManager(config_openai)

pregunta = [ChatMessage(role="user", content="¿Qué es la entropía? Respondé en 2 líneas.")]

resultado = await manager_openai.generate(pregunta)
print("🟢 OpenAI:", resultado.content if not resultado.error else f"❌ {resultado.error}")

🟢 OpenAI: ❌ Error de la API de OpenAI: Error code: 401 - {'error': {'message': 'Incorrect API key provided: cndkcskNF. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


##**Prueba en modo normal (Anthropic)**

In [ ]:
config_anthropic = LLMConfig(
    provider=Provider.ANTHROPIC,
    model="claude-3-5-sonnet-20241022",
    anthropic_api_key=SecretStr(os.environ["ANTHROPIC_API_KEY"]),
    temperature=0.7,
    max_tokens=200,
)

manager_anthropic = AsyncLLMManager(config_anthropic)

resultado = await manager_anthropic.generate(pregunta)
print("🟣 Anthropic:", resultado.content if not resultado.error else f"❌ {resultado.error}")

🟣 Anthropic: ❌ Error de la API de Anthropic: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CdgpZsLbwXuJoG28oEHDU'}


## **Prueba en modo streaming (los dos proveedores)**

El código que consume el stream es idéntico para ambos proveedores — eso es la abstracción funcionando.

In [ ]:
print("🟢 Streaming OpenAI:")
async for chunk in manager_openai.generate_stream(pregunta):
    print(chunk, end="", flush=True)

print("\n\n🟣 Streaming Anthropic:")
async for chunk in manager_anthropic.generate_stream(pregunta):
    print(chunk, end="", flush=True)

🟢 Streaming OpenAI:

[⚠️ Error durante el streaming: Error code: 401 - {'error': {'message': 'Incorrect API key provided: cndkcskNF. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}]

🟣 Streaming Anthropic:

[⚠️ Error durante el streaming: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CdgpaGztmYkxA3Nsf61L3'}]

## **Prueba de resiliencia (API key inválida a propósito)**

In [ ]:
config_rota = LLMConfig(
    provider=Provider.OPENAI,
    model="gpt-4o-mini",
    openai_api_key=SecretStr("sk-key-invalida-a-proposito"),
)

manager_roto = AsyncLLMManager(config_rota)
resultado = await manager_roto.generate(pregunta)

print("¿El programa siguió vivo?: ✅ Sí")
print("Error capturado (sin crash):", resultado.error)

¿El programa siguió vivo?: ✅ Sí
Error capturado (sin crash): Error de la API de OpenAI: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-key-i***************sito. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


In [ ]:
config_gemini = LLMConfig(
    provider=Provider.GEMINI,
    model="gemini-flash-latest",  # mismo alias estable que usamos en el proyecto RAG
    google_api_key=SecretStr(os.environ["GOOGLE_API_KEY"]),
    temperature=0.7,
    max_tokens=200,
)

manager_gemini = AsyncLLMManager(config_gemini)

resultado = await manager_gemini.generate(pregunta)
print("🔵 Gemini:", resultado.content if not resultado.error else f"❌ {resultado.error}")

print("\n🔵 Streaming Gemini:")
async for chunk in manager_gemini.generate_stream(pregunta):
    print(chunk, end="", flush=True)

🔵 Gemini: La entropía es la medida del

🔵 Streaming Gemini:
La entropía es una magnitud física que